# Hiệu năng pipeline

## Phạm vi phép đo

Nhóm đo pipeline tích hợp trên Google Colab CPU-only. Các phiên đã lưu có phạm vi đầu vào và lượng cache khác nhau; kết quả được đọc theo phiên thực thi, không dùng để xếp hạng thành viên. Các số đo lịch sử vẫn là bằng chứng hiệu năng của hệ thống và được tách khỏi kiểm kê corpus cuối.

Wall time tính thời gian end-to-end của lệnh pipeline. Timing từng paper và Discovery ghi thời gian các event đã hoàn tất. RSS cộng trên cây tiến trình; dung lượng output đo theo các lần lấy mẫu và ở cuối phiên.

In [ ]:
from pathlib import Path
import sys,json,hashlib
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,HTML,Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'_config.yml').exists())
sys.path.insert(0,str(ROOT/'utils'))
from preliminary_results import load_members,corpus_metrics,session_members,run_range,paper_timings
folder=ROOT/'summaries/milestone1/frozen-20261127'
receipt=json.loads((folder/'provenance.json').read_text(encoding='utf-8'))
raw=(folder/'corpus.json').read_bytes()
assert hashlib.sha256(raw).hexdigest()==receipt['public_sha256']
papers=json.loads(raw)['papers']
assert len(papers)==25000 and len({p['paper_id'] for p in papers})==25000
members=load_members(ROOT)
sessions=session_members(ROOT,members)
measurement=corpus_metrics(members)[-1]
def table(rows):
    frame=pd.DataFrame(rows,dtype=object).where(lambda f:f.notna(),'Không áp dụng')
    def fmt(v):
        if isinstance(v,int) and not isinstance(v,bool):return f'{v:,}'.replace(',','.')
        if isinstance(v,float):return f'{v:,.2f}'.replace(',','_').replace('.',',').replace('_','.')
        return str(v)
    display(HTML(frame.map(fmt).to_html(index=False,escape=True,border=0)))


In [ ]:
table([{'Phiên đo':i+1,'Ngày bắt đầu (UTC)':s['benchmark']['started_at'][:10],
        'Dải đầu vào':run_range(s['benchmark']),'Chế độ':'Resume',
        'Wall time (giây)':s['benchmark']['wall_seconds'],'Số mẫu RAM':s['benchmark']['sample_count']}
       for i,s in enumerate(sessions)])

## Thời gian Discovery và xử lý paper

Các run ID được khử trùng lặp. Việc một paper được xử lý lại ở hai phiên là chi phí thực, nhưng tổng các phiên hiện có không thay cho tổng lịch sử thu thập. Không cộng thời gian của các loại event chồng lặp thành wall time.

In [ ]:
rows=[]
for i,s in enumerate(sessions):
    events=[json.loads(l) for l in (s['directory']/'timings.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
    ds=[e for e in events if e['kind']=='discovery']; ps=[e for e in events if e['kind']=='paper']
    rows.append({'Phiên đo':i+1,'Tổng Discovery event (giây)':sum(e['elapsed_seconds'] for e in ds) if ds else None,
        'Paper event':len(ps),'Trung bình giây/paper event':sum(e['elapsed_seconds'] for e in ps)/len(ps) if ps else None})
table(rows)
frame=pd.DataFrame(paper_timings(sessions))
p=ROOT/'_static/generated/ms1-paper-timings.csv';p.parent.mkdir(parents=True,exist_ok=True);frame.to_csv(p,index=False)
display(HTML('<a href="../_static/generated/ms1-paper-timings.csv" download>Tải thời gian từng paper theo run ID (CSV)</a>'))

## RAM và dung lượng output

RAM trung bình cần có khoảng thời gian quan sát; phiên có 0–1 mẫu được đánh dấu không tính. Peak là cực đại quan sát, có thể bỏ lỡ đỉnh ngắn. Disk bao gồm raw source, corpus và các tệp trạng thái tích lũy; không cộng các phiên chồng lặp để suy dung lượng corpus.

In [ ]:
rows=[]
for i,s in enumerate(sessions):
    b=s['benchmark'];r=b['ram'];d=b['disk'];ok=b['sample_count']>=2 and r['observed_seconds']>0
    rows.append({'Phiên đo':i+1,'Peak RAM (MiB)':r['peak_rss_bytes']/1024**2 if ok else None,
        'RAM trung bình (MiB)':r['time_weighted_mean_rss_bytes']/1024**2 if ok and r['time_weighted_mean_rss_bytes'] is not None else None,
        'Peak output quan sát (GiB)':d['peak_observed_bytes']/1024**3 if d.get('peak_observed_bytes') is not None else None,
        'Output cuối (GiB)':d['final_bytes']/1024**3 if d.get('final_bytes') is not None else None})
table(rows)
chosen=next(s for s in sessions if s['benchmark']['sample_count']>2)
samples=[json.loads(l) for l in (chosen['directory']/'samples.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
fig,ax=plt.subplots(figsize=(8,3))
ax.plot([v['elapsed_seconds']/60 for v in samples],[v['rss_bytes']/1024**2 for v in samples])
ax.set_xlabel('Phút quan sát');ax.set_ylabel('RSS cây tiến trình (MiB)');ax.set_title('Diễn biến RAM trong một phiên đo có nhiều mẫu')
plt.tight_layout();plt.show()

Chi phí mạng, retry và quét thư mục Drive ảnh hưởng wall time và nhịp lấy mẫu. Resume có thể nhanh nhờ cache; các phép đo này không được ngoại suy thành tốc độ tải mới 25.000 paper. Không tái tạo RAM hoặc disk peak của những khoảng không được đo từ corpus cuối.